# Z-Score Normalization

---

## Pengertian

**Z-Score Normalization** (juga disebut **standardisasi**) adalah metode normalisasi yang mengubah nilai data sehingga menunjukkan seberapa jauh suatu nilai dari **rata-rata data** dalam satuan **standar deviasi**.

Berbeda dengan Min-Max Normalization yang sensitif terhadap outlier, Z-Score Normalization lebih **tahan terhadap outlier** karena tidak bergantung pada nilai minimum dan maksimum, melainkan pada rata-rata dan standar deviasi.

Setelah dinormalisasi, data akan memiliki:
- **Rata-rata (mean) = 0**
- **Standar deviasi = 1**

## Rumus

$$z = \frac{x - \mu}{\sigma}$$

**Keterangan:**
- $x$ : nilai asli dari data
- $\mu$ : rata-rata (mean) dari atribut
- $\sigma$ : standar deviasi dari atribut
- $z$ : nilai Z-Score (nilai setelah normalisasi)

## Rumus Rata-Rata

$$\mu = \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

## Rumus Standar Deviasi Populasi

$$\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(x_i - \mu)^2}$$

> **Catatan:** Terdapat dua jenis standar deviasi:
> - **Populasi** (ddof=0): dibagi dengan $N$ — digunakan jika data adalah seluruh populasi
> - **Sampel** (ddof=1): dibagi dengan $N-1$ — digunakan jika data adalah sampel dari populasi

## Variasi: Z-Score Menggunakan MAD

Selain menggunakan standar deviasi, terdapat variasi Z-Score menggunakan **Mean Absolute Deviation (MAD)** yang lebih tahan terhadap outlier.

$$s_A = \frac{1}{n}\sum_{i=1}^{n}|v_i - \bar{A}|$$

$$v' = \frac{v - \bar{A}}{s_A}$$

## Kelebihan dan Kekurangan

| Kelebihan | Kekurangan |
|-----------|------------|
| Lebih tahan terhadap outlier | Tidak memiliki batas atas/bawah yang pasti |
| Cocok untuk algoritma yang mengasumsikan distribusi normal | Hasil sulit diinterpretasikan secara langsung |
| Berguna untuk PCA dan regresi | Memerlukan mean dan std yang akurat |

## Dataset

| No | IPK | PO        | JML |
|----|-----|-----------|-----|
| 1  | 2   | 2.000.000 | 2   |
| 2  | 3   | 3.000.000 | 3   |
| 3  | 4   | 2.000.000 | 2   |
| 4  | 2   | 2.000.000 | 3   |
| 5  | 3   | 3.000.000 | 2   |
| 6  | 4   | 4.000.000 | 3   |

## Contoh Perhitungan Manual — Kolom IPK

Nilai IPK = {2, 3, 4, 2, 3, 4}

**Langkah 1 — Hitung Mean:**

$$\mu = \frac{2+3+4+2+3+4}{6} = \frac{18}{6} = 3$$

**Langkah 2 — Hitung Standar Deviasi Populasi:**

$$\sigma = \sqrt{\frac{(2-3)^2+(3-3)^2+(4-3)^2+(2-3)^2+(3-3)^2+(4-3)^2}{6}}$$

$$= \sqrt{\frac{1+0+1+1+0+1}{6}} = \sqrt{\frac{4}{6}} = \sqrt{0.6667} \approx 0.8165$$

**Langkah 3 — Hitung Z-Score setiap objek:**

$$z_1 = \frac{2 - 3}{0.8165} = \frac{-1}{0.8165} \approx -1.2247$$

$$z_2 = \frac{3 - 3}{0.8165} = \frac{0}{0.8165} = 0$$

$$z_3 = \frac{4 - 3}{0.8165} = \frac{1}{0.8165} \approx 1.2247$$

$$z_4 = \frac{2 - 3}{0.8165} \approx -1.2247, \quad z_5 = 0, \quad z_6 \approx 1.2247$$

## Implementasi Python — Fungsi Manual

In [ ]:
import pandas as pd
import numpy as np

# Dataset
data = {
    'No':  [1, 2, 3, 4, 5, 6],
    'IPK': [2, 3, 4, 2, 3, 4],
    'PO':  [2000000, 3000000, 2000000, 2000000, 3000000, 4000000],
    'JML': [2, 3, 2, 3, 2, 3]
}
df = pd.DataFrame(data).set_index('No')

print("=== Data Sebelum Normalisasi ===")
print(df)

In [ ]:
# =========================================
# Fungsi Z-Score Normalization Manual
# =========================================
def zscore_normalization(df, kolom, ddof=0):
    """
    Melakukan Z-Score Normalization pada kolom yang dipilih.
    
    Parameters:
        df    : DataFrame
        kolom : list kolom yang akan dinormalisasi
        ddof  : derajat kebebasan (0 = populasi, 1 = sampel)
    
    Returns:
        DataFrame hasil normalisasi
    """
    df_out = df.copy()
    for col in kolom:
        mu    = df[col].mean()
        sigma = df[col].std(ddof=ddof)
        df_out[col] = (df[col] - mu) / sigma
        print(f"Kolom {col}: μ = {mu:.4f}, σ = {sigma:.6f}")
    return df_out

kolom_target = ['IPK', 'PO', 'JML']
df_zscore = zscore_normalization(df, kolom_target, ddof=0)

print("\n=== Hasil Z-Score Normalization (Fungsi Manual) ===")
print(df_zscore.round(6))

In [ ]:
# Detail perhitungan per objek untuk kolom IPK
print("=== Detail Perhitungan Z-Score Kolom IPK ===")
mu_ipk    = df['IPK'].mean()
sigma_ipk = df['IPK'].std(ddof=0)
print(f"Mean (μ)   = {mu_ipk}")
print(f"Std Dev (σ) = {sigma_ipk:.6f}\n")

print(f"{'No':<5} {'IPK':<8} {'(x - μ)':<12} {'z = (x-μ)/σ':<15}")
print("-" * 45)
for idx, val in df['IPK'].items():
    diff = val - mu_ipk
    z    = diff / sigma_ipk
    print(f"{idx:<5} {val:<8} {diff:<12.4f} {z:<15.6f}")

In [ ]:
# =========================================
# Variasi: Z-Score Menggunakan MAD
# =========================================
def zscore_mad(df, kolom):
    """
    Z-Score Normalization menggunakan Mean Absolute Deviation (MAD).
    Lebih tahan terhadap outlier dibandingkan standar deviasi.
    """
    df_out = df.copy()
    for col in kolom:
        mu  = df[col].mean()
        mad = (df[col] - mu).abs().mean()
        df_out[col] = (df[col] - mu) / mad
        print(f"Kolom {col}: μ = {mu:.4f}, MAD = {mad:.6f}")
    return df_out

df_mad = zscore_mad(df, kolom_target)

print("\n=== Z-Score Menggunakan MAD (Fungsi Manual) ===")
print(df_mad.round(4))

## Implementasi Python — Menggunakan sklearn

In [ ]:
from sklearn.preprocessing import StandardScaler

# =========================================
# StandardScaler dari sklearn
# =========================================
# Catatan: sklearn StandardScaler menggunakan ddof=0 (standar deviasi populasi)

scaler = StandardScaler()

df_sklearn = df.copy()
df_sklearn[kolom_target] = scaler.fit_transform(df[kolom_target])

print("=== Hasil Z-Score Normalization (sklearn StandardScaler) ===")
print(df_sklearn.round(6))

print("\nMean per kolom (dari scaler):", scaler.mean_)
print("Std per kolom (dari scaler) :", scaler.scale_)

In [ ]:
# =========================================
# Verifikasi & Perbandingan
# =========================================
print("=== Verifikasi: Manual vs sklearn ===")
print(f"Hasil identik: {df_zscore.round(6).equals(df_sklearn.round(6))}")

print("\n=== Perbandingan Z-Score vs Z-Score MAD (Kolom IPK) ===")
print(pd.DataFrame({
    'Original'    : df['IPK'],
    'Z-Score Std' : df_zscore['IPK'].round(4),
    'Z-Score MAD' : df_mad['IPK'].round(4)
}))

print("\nStatistik setelah Z-Score Std:")
print(f"  Mean = {df_zscore['IPK'].mean():.6f}  (harusnya ≈ 0)")
print(f"  Std  = {df_zscore['IPK'].std(ddof=0):.6f}  (harusnya ≈ 1)")

## Kesimpulan

- Z-Score Normalization mengubah data sehingga memiliki **mean = 0** dan **standar deviasi = 1**
- Rumus: $z = (x - \mu) / \sigma$
- Lebih tahan terhadap **outlier** dibandingkan Min-Max Normalization
- Cocok untuk algoritma yang mengasumsikan distribusi normal seperti **regresi linear, PCA, SVM**
- Variasi menggunakan **MAD** memberikan hasil yang lebih robust terhadap data ekstrem
- Tidak ada batas rentang yang pasti (tidak terbatas pada [0, 1])